# Building a Knowledge Graph based on Databricks Data

**Author:** Niklas Wagner  
**Acknowledgements:** Thanks to Kenneth Leung for the initial notebook and guidance  

---

This notebook demonstrates how to extract metadata from Databricks tables and construct a knowledge graph.  
It includes steps to:

1. Connect to Databricks SQL  
2. Extract table and column metadata  
3. Build a consolidated metadata DataFrame   
4. Load the data into a graph database (Neo4j) for knowledge graph exploration

The workflow is based on an initial notebook by Kenneth Leung, adapted and extended for graph construction from databricks data.

In [6]:
import datetime
import os
import sys


from dotenv import load_dotenv
from neo4j import GraphDatabase
from neo4j_graphrag.retrievers import HybridCypherRetriever
from openai import AzureOpenAI
from pprint import pprint
import numpy as np
import yaml

import os
import pandas as pd
from databricks import sql

sys.path.append("../")

from src.utils.utils import load_config

load_dotenv()

True

In [7]:
config = load_config("src/config/config.yaml")
config

{'KG_SCHEMA_CONCEPTS_FILE': 'data/concepts.yaml',
 'OPENAI_EMBEDDING_MODEL': 'text-embedding-3-large',
 'TABLE_VECTOR_INDEX_NAME': 'table_vector_index',
 'TABLE_VECTOR_INDEX_INPUT_PROPERTIES': ['title', 'description', 'remarks'],
 'TABLE_FULLTEXT_INDEX_NAME': 'table_fulltext_index',
 'TABLE_FULLTEXT_INPUT_PROPERTIES': ['title', 'description'],
 'TABLE_EMBEDDING_PROPERTY_NAME': 'table_embedding',
 'EMBEDDING_DIMENSIONS': 3072,
 'LLM_MODEL': 'gpt-4o-mini',
 'TEMPERATURE': 0,
 'SEED': 42,
 'TOP_K': 3}

### Setup AzureOpenAI client

**Note:** This setup is specific to Azure OpenAI and may vary for other models or providers.


In [8]:
client = AzureOpenAI(
    api_key=os.environ["AZURE_OPENAI_API_KEY"],
    api_version=os.environ["AZURE_OPENAI_API_VERSION"],
    base_url=os.environ["AZURE_OPENAI_BASE_URL"],
)

___
### Connect to Databricks

In [9]:
# --- Extract Databricks connection info from environment ---
host = os.environ.get("DATABRICKS_HOST")
token = os.environ.get("DATABRICKS_TOKEN")
warehouse_id = os.environ.get("DATABRICKS_WAREHOUSE_ID")
catalog = os.environ.get("DATABRICKS_CATALOG")
schema = os.environ.get("DATABRICKS_SCHEMA")

# --- Assertions to make sure the variables exist ---
assert host, "DATABRICKS_HOST environment variable is not set"
assert token, "DATABRICKS_TOKEN environment variable is not set"
assert warehouse_id, "DATABRICKS_WAREHOUSE_ID environment variable is not set"
assert catalog, "DATABRICKS_CATALOG environment variable is not set"
assert schema, "DATABRICKS_SCHEMA environment variable is not set"

# --- Connect to Databricks SQL ---
conn = sql.connect(
    server_hostname=host,
    http_path=f"/sql/1.0/warehouses/{warehouse_id}",
    access_token=token,
)

# --- Info about the data ---
print(f"Connected to Databricks SQL at {host}")
print(f"Catalog: {catalog}, Schema: {schema}")
print("Metadata about tables and columns in this schema can now be queried for building a knowledge graph or data catalog.")

Connected to Databricks SQL at dbc-66c192e9-609f.cloud.databricks.com
Catalog: alfred_app, Schema: finance
Metadata about tables and columns in this schema can now be queried for building a knowledge graph or data catalog.


In [10]:
query = f"""
SELECT
    c.table_name,
    c.column_name,
    c.data_type,

    CASE 
        WHEN t.constraint_type = 'PRIMARY KEY' THEN 'yes'
        ELSE 'no'
    END AS is_primary_key,

    c.comment AS column_description,
    NULL AS column_remarks,
    NULL AS column_sample_value,
    NULL AS common_joins_x,

    tbl.comment AS table_title,
    tbl.comment AS table_description,

    '{schema}' AS table_cluster_name,
    NULL AS source_filename,
    NULL AS table_remarks,
    'active' AS deprecation_status,
    NULL AS common_joins_y,
    NULL AS source_filename_1,
    '{schema}' AS table_cluster_title,
    NULL AS table_cluster_description,

    c.table_name AS included_tables,
    NULL AS user_group,
    NULL AS user_group_description

FROM {catalog}.information_schema.columns c

LEFT JOIN {catalog}.information_schema.tables tbl
    ON c.table_catalog = tbl.table_catalog
    AND c.table_schema = tbl.table_schema
    AND c.table_name = tbl.table_name

LEFT JOIN {catalog}.information_schema.key_column_usage k
    ON c.table_catalog = k.table_catalog
    AND c.table_schema = k.table_schema
    AND c.table_name = k.table_name
    AND c.column_name = k.column_name

LEFT JOIN {catalog}.information_schema.table_constraints t
    ON k.constraint_name = t.constraint_name
    AND t.constraint_type = 'PRIMARY KEY'

WHERE c.table_schema = '{schema}'
ORDER BY c.table_name, c.ordinal_position
"""

### Review Schema Information

In [11]:
from databricks import sql
import pandas as pd
import os

conn = sql.connect(
    server_hostname=os.environ["DATABRICKS_HOST"],
    http_path=f"/sql/1.0/warehouses/{os.environ['DATABRICKS_WAREHOUSE_ID']}",
    access_token=os.environ["DATABRICKS_TOKEN"],
)

with conn.cursor() as cursor:
    cursor.execute(query)
    rs = cursor.fetchall_arrow()  # returns a pyarrow.Table
    df_db_details = rs.to_pandas()  # pyarrow -> pandas

display(df_db_details.head())

,table_name,column_name,data_type,is_primary_key,column_description,column_remarks,column_sample_value,common_joins_x,table_title,table_description,...,source_filename,table_remarks,deprecation_status,common_joins_y,source_filename_1,table_cluster_title,table_cluster_description,included_tables,user_group,user_group_description
0,account,account_id,LONG,no,A unique identifier for each financial account...,NaN,NaN,NaN,The table contains information related to fina...,The table contains information related to fina...,...,NaN,NaN,active,NaN,NaN,finance,NaN,account,NaN,NaN
1,account,district_id,LONG,no,Identifies the district associated with the fi...,NaN,NaN,NaN,The table contains information related to fina...,The table contains information related to fina...,...,NaN,NaN,active,NaN,NaN,finance,NaN,account,NaN,NaN
2,account,frequency,STRING,no,"Indicates how often account activities occur, ...",NaN,NaN,NaN,The table contains information related to fina...,The table contains information related to fina...,...,NaN,NaN,active,NaN,NaN,finance,NaN,account,NaN,NaN
3,account,date,LONG,no,Represents the relevant date associated with t...,NaN,NaN,NaN,The table contains information related to fina...,The table contains information related to fina...,...,NaN,NaN,active,NaN,NaN,finance,NaN,account,NaN,NaN
4,card,card_id,LONG,no,"A unique identifier for each card issued, allo...",NaN,NaN,NaN,The table contains information about various c...,The table contains information about various c...,...,NaN,NaN,active,NaN,NaN,finance,NaN,card,NaN,NaN


___
### Consolidate Information from Data Dictionary
- Aim is to create a consolidated masterlist of all fields across all tables

___
### Setup and Review Concepts
- The idea of Concepts is to act as a bridge between relational data and semantic meaning, foundational for building a graph-based abstraction over structured databases
- The YAML file contains information to do the following:
    - Define **core semantic entities or attributes** used in the knowledge graph:
        - E.g., AccountIdentifier, MonetaryValue, TransactionCategory
        - Each has a description and example value for context
        - Used to abstract raw column values into meaningful business concepts
    - Describe **semantic relationships between concepts** for the knowledge graph:
        - Defines how concepts are logically connected (e.g., CustomerIdentifier → AccountIdentifier)
        - Each has a relationship type (e.g., IS_ASSOCIATED_WITH) and a description
        - Supports graph reasoning and natural language query generation
    - Map **raw SQL table columns to defined semantic concepts**:
        - Supports translation from SQL schema to conceptual schema
        - Includes:
            - global: mappings that apply across all tables
            - table-specific mappings (e.g., client.birth_number → DerivedGender)
        - Enables semantic enrichment of SQL data in the graph

In [ ]:
with open('./concepts/concepts.yaml', 'r') as file:
    yaml_content = yaml.safe_load(file)
pprint(yaml_content)

{'column_concept_mapping': {'card': {'issued': 'Date'},
                            'client': {'birth_number': 'DerivedGender'},
                            'district': {'A1': 'DistrictIdentifier'},
                            'global': {'CustomerID': 'CustomerIdentifier',
                                       'account_id': 'AccountIdentifier',
                                       'amount': 'MonetaryValue',
                                       'balance': 'MonetaryValue',
                                       'bank': 'BankIdentifier',
                                       'birth_number': 'Date',
                                       'date': 'Date',
                                       'district_id': 'DistrictIdentifier',
                                       'k_symbol': 'TransactionCategory'},
                            'order': {'account_to': 'AccountIdentifier',
                                      'bank_to': 'BankIdentifier'},
                            'trans': {'accou

___
## Setup Neo4j Instance (Local via Docker)
1) Make sure Docker is installed and running on your system. Check that it is running with `docker --version` in Terminal
2) Download the official Neo4j image from Docker Hub with `docker pull neo4j:latest`
3) Run the Neo4j instance locally with the following command:
`docker run --name neo4j -p 7474:7474 -p 7687:7687 -e NEO4J_AUTH=neo4j/password neo4j`

Details:
- `--name neo4j`: Sets the container name to `neo4j`.
- `-p 7474:7474`: Maps the Neo4j HTTP interface to port `7474` on your host machine.
- `-p 7687:7687`: Maps the Neo4j Bolt protocol to port `7687` on your host machine.
- `-e NEO4J_AUTH=neo4j/password`: Sets the authentication credentials (default username is `neo4j` and password is `password`).
- `-d`: Runs the container in detached mode (in the background).
- `neo4j:latest`: Uses the latest version of the Neo4j image.

4) Open your browser and go to http://localhost:7474 , and log in using the credentials you set (username is `neo4j` and password is `password`).
5) (Optional) Persist Data with Volumes: To ensure that your data is not lost when the container stops, you can mount volumes for Neo4j's data, logs, and configuration files:
`docker run --name neo4j -p 7474:7474 -p 7687:7687 -e NEO4J_AUTH=neo4j/password -v $HOME/neo4j/data:/data -v $HOME/neo4j/logs:/logs -v $HOME/neo4j/conf:/conf -d neo4j:latest`
:latest

___
### Restart existing Neo4j Instance
1) Verify Docker is running using `docker ps -a` (Start Docker Desktop first)
2) Start the existing Neo4j container: `docker start neo4j`
3) After starting the container, you can access Neo4j in your browser at `http://localhost:7474` using the previously set credentials (`neo4j` / `password`).

___
### Setup Driver
- Initializes a connection to a Neo4j graph database using the Bolt protocol
    - Retrieves the database URL, username, and password from environment variables to securely authenticate and create a driver object for executing Cypher queries

In [13]:
driver = GraphDatabase.driver(os.getenv('NEO4J_BOLT_URL'), 
                              auth=(os.getenv('NEO4J_USERNAME'), 
                                    os.getenv('NEO4J_PASSWORD')
                                   ))

# Flush the graph database
with driver.session() as session:
    # Delete all nodes and relationships
    session.run("MATCH (n) DETACH DELETE n")

___
### Build Nodes and Relationships for SQL Tables
- Loads schema metadata and concept mappings from YAML file
- Connects to Neo4j and clears the graph (optional)
- Converts a schema DataFrame into a property graph (A property graph stores structured, labeled, and attributed data as nodes and relationships, allowing for rich querying and flexible schema)
    - Creates Table, Column, and TableCluster nodes
    - Links columns to tables, and tables to clusters
    - Maps columns to Concept nodes using YAML mappings
- Establishes relationships between Concept nodes (e.g., semantic links).
- Generates and stores OpenAI embeddings for Table nodes.
- Creates vector and fulltext indexes on Table nodes for semantic and keyword search.

In [14]:
def make_serializable(obj):
    """
    Utility function that can handle non-serializable types (e.g., numpy int64, etc.).
    Convert them to Python native types that can be passed safely to Neo4j queries.
    """
    if isinstance(obj, dict):
        return {key: make_serializable(value) for key, value in obj.items()}
    elif isinstance(obj, list):
        return [make_serializable(item) for item in obj]
    elif isinstance(obj, np.generic):  # Handle NumPy scalar types
        return obj.item()
    elif isinstance(obj, (pd.Timestamp, datetime.datetime)):  # Handle datetime and pd.Timestamp
        return obj.isoformat()
    elif isinstance(obj, datetime.time):  # Handle time objects
        return obj.isoformat()  # Convert to ISO 8601 string
    else:
        return obj

In [15]:
class DatabaseSchemaToGraph:
    def __init__(self, concepts_filepath):
        self.driver = GraphDatabase.driver(os.getenv('NEO4J_BOLT_URL'), 
                              auth=(os.getenv('NEO4J_USERNAME'), 
                                    os.getenv('NEO4J_PASSWORD')
                                   ))
        (
            self.column_concept_mapping, 
            self.concept_definitions, 
            self.concept_relationships
        ) = self.load_concept_mapping(concepts_filepath)

    def clear_graph(self):
        """Deletes all existing nodes and relationships from the graph."""
        with self.driver.session() as session:
            session.run("MATCH (n) DETACH DELETE n")

    @staticmethod
    def load_concept_mapping(concept_mapping_file):
        """Load the concept mapping, concept definitions, and concept relationships from a YAML file."""
        with open(concept_mapping_file, 'r', encoding='utf-8') as file:
            mapping = yaml.safe_load(file)
        concept_mapping = mapping.get('column_concept_mapping', {})
        concept_definitions = mapping.get('concepts', {})
        concept_relationships = mapping.get('concept_relationships', [])
        return concept_mapping, concept_definitions, concept_relationships

    def ingest_schema(self, df):
        """
        Orchestrates the ingestion of the schema into Neo4j.
        Clears the graph (if you want a fresh start) and then:
          1) Creates/merges table nodes
          2) Creates/merges column nodes and links them to the tables
          3) Links columns to concept nodes
        """
        self.clear_graph()  # remove this if you don't want to wipe the DB every time
        self.create_table_nodes(df)
        self.create_column_nodes(df)
        self.create_table_cluster_nodes(df)
        self.link_columns_to_concepts(df)

    def create_table_nodes(self, df):
        """
        Creates/merges Table nodes. Also sets table-level attributes,
        including primary keys (as a list) and any other metadata.
        """
        with self.driver.session() as session:
            for table_name, group in df.groupby('table_name'):
                # Extract table-level attributes
                description = group['table_description'].iloc[0]
                cluster = group['table_cluster_name'].iloc[0]
                title = group['table_title'].iloc[0]
                remarks = group['table_remarks'].iloc[0]
                source_filename = group['source_filename'].iloc[0]

                # Identify primary key columns
                primary_keys = (
                    group.loc[group['is_primary_key'].str.lower() == 'yes', 'column_name']
                    .tolist()
                )

                # Merge the Table node
                session.run(
                    """
                    MERGE (t:Table {name: $name})
                    SET t.title = $title,
                        t.description = $description,
                        t.cluster = $cluster,
                        t.remarks = $remarks,
                        t.source_filename = $source_filename,
                        t.primary_keys = $primary_keys
                    """,
                    name=table_name,
                    title=title,
                    description=description,
                    cluster=cluster,
                    remarks=remarks,
                    source_filename=source_filename,
                    primary_keys=primary_keys
                )

    def create_column_nodes(self, df):
        """
        Creates/merges Column nodes for each table, linking them with a CONTAINS relationship:
            (Table)-[:CONTAINS]->(Column)
        """
        with self.driver.session() as session:
            for table_name, group in df.groupby('table_name'):
                # We'll assume the Table node is already created by create_table_nodes
                # Just attach columns now
                cols_to_keep = [
                    'column_name', 'data_type', 'is_primary_key',
                    'column_remarks', 'column_sample_value'
                ]
                for _, row in group.iterrows():
                    column_name = row['column_name']
                    
                    # Gather relevant column properties
                    column_properties = row[cols_to_keep].dropna().to_dict()
                    column_properties['description'] = row.get('column_description', None)
                    column_properties = make_serializable(column_properties)

                    session.run(
                        """
                        MATCH (t:Table {name: $table_name})
                        MERGE (c:Column {name: $column_name, table_name: $table_name})
                        SET c += $properties
                        MERGE (t)-[:CONTAINS]->(c)
                        """,
                        table_name=table_name,
                        column_name=column_name,
                        properties=column_properties
                    )


    def create_table_cluster_nodes(self, df):
        """
        Creates/merges TableCluster nodes from the DataFrame and links them 
        to associated Table nodes via the CONTAINS relationship.
        """
        with self.driver.session() as session:
            # We only need the columns relevant to clusters; drop duplicates just in case.
            cluster_df = df[['table_cluster_name', 'table_cluster_description', 'table_cluster_title', 
                             'included_tables']].drop_duplicates()
    
            for _, row in cluster_df.iterrows():
                cluster_name = row['table_cluster_name']
                cluster_title = row['table_cluster_title']
                cluster_description = row['table_cluster_description']
                included_tables = row['included_tables']
    
                # Convert the comma-separated table names into a proper list.
                # e.g., "account, client" => ["account", "client"]
                table_list = [tbl.strip() for tbl in included_tables.split(',') if tbl.strip()]
    
                # Create / merge the cluster node and set its description
                session.run(
                    """
                    MERGE (tc:TableCluster {name: $cluster_name})
                    SET tc.description = $cluster_description
                    """,
                    cluster_name=cluster_name,
                    cluster_title=cluster_title,
                    cluster_description=cluster_description
                )
    
                # Now link the cluster to each table in the list.
                # We assume those Table nodes already exist from create_table_nodes(...).
                for tbl in table_list:
                    session.run(
                        """
                        MATCH (tc:TableCluster {name: $cluster_name})
                        MATCH (t:Table {name: $table_name})
                        MERGE (tc)-[:CONTAINS]->(t)
                        """,
                        cluster_name=cluster_name,
                        table_name=tbl
                    )


    def link_columns_to_concepts(self, df):
        """
        For each column, check if there's a concept mapping (table-specific or global).
        If so, merge the Concept node and create a MAPS_TO_CONCEPT relationship:
            (Column)-[:MAPS_TO_CONCEPT]->(Concept)
        """
        with self.driver.session() as session:
            for table_name, group in df.groupby('table_name'):
                # Get the column-to-concept mapping for this table
                table_mapping = self.column_concept_mapping.get(table_name, {})
                
                for _, row in group.iterrows():
                    column_name = row['column_name']
                    
                    # Determine the concept for this column in the context of the table
                    concept_name = table_mapping.get(column_name)

                    # If concept not found in the table-specific mapping, check global
                    if not concept_name:
                        global_mapping = self.column_concept_mapping.get('global', {})
                        concept_name = global_mapping.get(column_name)

                    if concept_name:
                        # Get concept properties from definitions
                        concept_properties = self.concept_definitions.get(concept_name, {})
                        concept_properties['name'] = concept_name
                        concept_properties = make_serializable(concept_properties)

                        session.run(
                            """
                            MERGE (con:Concept {name: $concept_name})
                            SET con += $concept_properties
                            MERGE (c:Column {name: $column_name, table_name: $table_name})
                            MERGE (c)-[:MAPS_TO_CONCEPT]->(con)
                            """,
                            concept_name=concept_name,
                            concept_properties=concept_properties,
                            column_name=column_name,
                            table_name=table_name
                        )

    def create_concept_relationships(self):
        """Creates relationships between Concept nodes based on the YAML configuration."""
        with self.driver.session() as session:
            for rel in self.concept_relationships:
                from_concept = rel['from']
                to_concept = rel['to']
                rel_type = rel['type']
                description = rel.get('description', None)

                from_properties = self.concept_definitions.get(from_concept, {})
                from_properties['name'] = from_concept
                from_properties = make_serializable(from_properties)

                to_properties = self.concept_definitions.get(to_concept, {})
                to_properties['name'] = to_concept
                to_properties = make_serializable(to_properties)

                session.run(
                    f"""
                    MERGE (from_con:Concept {{name: $from_concept}})
                    SET from_con += $from_properties
                    MERGE (to_con:Concept {{name: $to_concept}})
                    SET to_con += $to_properties
                    MERGE (from_con)-[r:{rel_type} {{description: $description}}]->(to_con)
                    """,
                    from_concept=from_concept,
                    from_properties=from_properties,
                    to_concept=to_concept,
                    to_properties=to_properties,
                    description=description
                )

    def generate_table_node_embeddings(self, 
                                       embedding_model=config['OPENAI_EMBEDDING_MODEL'], 
                                       properties=config['TABLE_VECTOR_INDEX_INPUT_PROPERTIES'], 
                                       embedding_property=config['TABLE_EMBEDDING_PROPERTY_NAME']):
        """
        Generate OpenAI embeddings for each Table node and store them on the node.
        
        :param embedding_model: The OpenAI engine/model to use for embeddings.
        :param properties: List of Table node properties to concatenate for embedding.
        :param embedding_property: The property name on the node to store the embedding.
        """
        with self.driver.session() as session:
            # Fetch all Table nodes and their textual properties
            query = f"""
            MATCH (t:Table)
            RETURN elementId(t) AS node_id, {", ".join([f"t.{p} AS {p}" for p in properties])}
            """
            results = session.run(query)
    
            # For each node, concatenate properties and embed
            for record in results:
                node_id = record["node_id"]
    
                # Concatenate the relevant text fields (filter out None or empty)
                texts = [str(record[prop]) if record[prop] is not None else "" for prop in properties]
                text_to_embed = " ".join(texts).strip()  # Trim leading/trailing spaces
    
                if not text_to_embed.strip():
                    # If there's no text, you might choose to skip or store an empty embedding
                    continue
    
                # Call OpenAI to get the embedding vector
                embedding_vector = client.embeddings.create(
                    input=[text_to_embed],  # single item in a list
                    model=embedding_model
                )               .data[0].embedding
                # Store the embedding back in Neo4j
                update_query = f"""
                MATCH (t:Table)
                WHERE elementId(t) = $node_id
                SET t.{embedding_property} = $embedding
                """
                session.run(update_query, node_id=node_id, embedding=embedding_vector)
            print(f"Embeddings stored in node property `{embedding_property}`.")
    
    def create_vector_index(self, 
                            index_name=config['TABLE_VECTOR_INDEX_NAME'], 
                            embedding_property=config['TABLE_EMBEDDING_PROPERTY_NAME'], 
                            dimensions=config['EMBEDDING_DIMENSIONS']):
        """
        Creates a Neo4j vector index for fast embedding-based similarity search.
        :param index_name: The name of the vector index.
        :param embedding_property: The node property storing the embeddings.
        :param dimensions: The number of dimensions in the embedding model.
        """
        query = f"""
        CREATE VECTOR INDEX {index_name}
        FOR (t:Table)
        ON (t.{embedding_property})
        OPTIONS {{
            indexConfig: {{
                `vector.dimensions`: {dimensions},
                `vector.similarity_function`: "cosine"
            }}
        }}
        """
        with self.driver.session() as session:
            session.run(f"DROP INDEX {index_name} IF EXISTS")  # Drop existing if needed
            session.run(query)
            print(f"Vector index `{index_name}` created on `{embedding_property}` with {dimensions} dimensions.")

    def create_fulltext_index(self, 
                              index_name=config['TABLE_FULLTEXT_INDEX_NAME'], 
                              properties=config['TABLE_FULLTEXT_INPUT_PROPERTIES']
                             ):
        """
        Creates a fulltext index for Table nodes in Neo4j on the specified properties.
    
        :param index_name: The name of the fulltext index.
        :param properties: List of Table node properties to include in the index.
        """
        # Build the property list for the Cypher query
        props_cypher = ", ".join(f"t.{prop}" for prop in properties)
    
        create_index_query = f"""
        CREATE FULLTEXT INDEX {index_name}
        FOR (t:Table)
        ON EACH [{props_cypher}]
        """
    
        with self.driver.session() as session:
            # Drop index if it already exists (to avoid conflicts)
            drop_query = f"DROP INDEX {index_name} IF EXISTS"
            session.run(drop_query)
    
            # Create the fulltext index
            session.run(create_index_query)
            print(f"Fulltext index `{index_name}` created on properties: {properties}")
    
    
    def run_cypher_query(self, query):
        with self.driver.session() as session:
            result = session.run(query)
            return [record for record in result]

In [16]:
neo4j_kg = DatabaseSchemaToGraph(
    concepts_filepath=f"./{config['KG_SCHEMA_CONCEPTS_FILE']}"
)
neo4j_kg.ingest_schema(df_db_details)  
neo4j_kg.create_concept_relationships()
neo4j_kg.generate_table_node_embeddings()
neo4j_kg.create_vector_index()  # Create the vector index
neo4j_kg.create_fulltext_index()

Embeddings stored in node property `table_embedding`.
Vector index `table_vector_index` created on `table_embedding` with 3072 dimensions.
Fulltext index `table_fulltext_index` created on properties: ['title', 'description']


In [17]:
query = "MATCH (n) RETURN n LIMIT 1"
records = neo4j_kg.run_cypher_query(query)
for record in records:
    print(record, "\n")

<Record n=<Node element_id='4:a26e0165-6bfa-44b1-8354-add040b5f7e5:206' labels=frozenset({'Table'}) properties={'table_embedding': [-0.00601477175951004, 0.030569011345505714, -0.014578240923583508, -0.01452450267970562, 0.012152371928095818, -0.03402357175946236, -0.023368174210190773, 0.012505505234003067, -0.01212166529148817, 0.017272796481847763, 0.05186445266008377, -0.017810173332691193, -0.021955644711852074, 0.003009305102750659, -0.034791249781847, -0.003648398444056511, -0.007799627259373665, -0.02209382690489292, 0.06559056788682938, 0.022830799221992493, 0.029893454164266586, 0.012336615473031998, -0.01103155966848135, -0.013173386454582214, 0.03497549518942833, 0.023276053369045258, -0.023552417755126953, -0.0027598091401159763, -0.03049224428832531, 0.010709133930504322, 0.01192206796258688, 0.016412995755672455, -0.0016917744651436806, -0.05647053197026253, -0.014363289810717106, -0.015737436711788177, -0.026807380840182304, 0.005926488433033228, 0.024074440822005272, 0

### View generated graph in Neo4j browser: http://localhost:7474/browser/ by running `MATCH (n) RETURN n`

___

In [18]:
# Create custom class for OpenAI embeddings
class OpenAIEmbedder:
    def __init__(self, model=config["OPENAI_EMBEDDING_MODEL"], api_key=None):
        self.model = model
        if api_key:
            AzureOpenAI.api_key = api_key
    def embed_query(self, text):
        response = client.embeddings.create(
            input=text,
            model=self.model
        )
        return response.data[0].embedding

### Setup standardized retrieval query for graph traversal
- The Cypher query retrieves `Table` nodes along with their columns, any associated `Concept` nodes, and related `Column` nodes linked via those `Concepts`.
- It also extracts detailed metadata for each column, including its name, description, data type, and a sample value.
- This enriched information helps facilitate text-to-SQL generation by providing structured context about the database schema

In [19]:
# Setup Cypher query to do retrieval of necessary context from the KG
retrieval_query = """
MATCH (node:Table)
OPTIONAL MATCH (node)-[:CONTAINS]->(col:Column)
OPTIONAL MATCH (col)-[:MAPS_TO_CONCEPT]->(concept:Concept)
OPTIONAL MATCH (concept)<-[:MAPS_TO_CONCEPT]-(relatedCol:Column)

RETURN 
  node.name AS table_name,
  node.description AS table_description,
  
  // Only include columns if col.name is not null
  collect(DISTINCT CASE 
    WHEN col.name IS NOT NULL THEN {
      column_name: col.name, 
      description: COALESCE(col.description, "No description available"), 
      data_type: COALESCE(col.data_type, "Unknown"), 
      column_sample_value: COALESCE(col.column_sample_value, "No sample value")
    } 
  END) AS columns,

  collect(DISTINCT concept.name) AS concepts,
  
  // Only include related columns if relatedCol.name is not null
  collect(DISTINCT CASE 
    WHEN relatedCol.name IS NOT NULL THEN {
      column_name: relatedCol.name, 
      description: COALESCE(relatedCol.description, "No description available"), 
      data_type: COALESCE(relatedCol.data_type, "Unknown"), 
      column_sample_value: COALESCE(relatedCol.column_sample_value, "No sample value")
    } 
  END) AS related_columns
"""

___
### Setup Hybrid Retriever
Creates a hybrid retriever that combines vector search and graph (Cypher) query capabilities to retrieve relevant information from your Neo4j knowledge graph in response to a user query

In [20]:
retriever = HybridCypherRetriever(
    driver=driver,
    vector_index_name=config['TABLE_VECTOR_INDEX_NAME'],
    fulltext_index_name=config['TABLE_FULLTEXT_INDEX_NAME'],
    embedder=OpenAIEmbedder(),
    retrieval_query=retrieval_query
)

___
### Run retrieval based on user query

In [21]:
# Enter user query
query_text = "Which district has the most customers staying in it?"

In [22]:
retriever_result = retriever.search(query_text=query_text, top_k=config["TOP_K"])
output_lines = []

for item in retriever_result.items:
    raw_text = item.content
    if raw_text.startswith("<Record "):
        raw_text = raw_text[len("<Record "):]
    if raw_text.endswith(">"):
        raw_text = raw_text[:-1]

    # Normalize extra whitespace
    raw_text = " ".join(raw_text.split())

    # Add the cleaned line to the list
    output_lines.append(raw_text)
    output_lines.append("\n")

retrieved_contents = "\n".join(output_lines)
print(retrieved_contents)

table_name='order' table_description='The table contains data on financial orders, which includes details such as the order identifier, associated accounts, and transaction amounts. This information can be utilized for tracking financial activities, auditing transactions, and analyzing order patterns across different accounts.' columns=[{'column_name': 'k_symbol', 'data_type': 'STRING', 'description': 'Serves as a symbol or code that may represent a specific financial instrument or asset related to the order, aiding in classification.', 'column_sample_value': 'No sample value'}, {'column_name': 'amount', 'data_type': 'DOUBLE', 'description': 'Denotes the total value of the financial order, representing the transaction sum associated with that order.', 'column_sample_value': 'No sample value'}, {'column_name': 'account_to', 'data_type': 'LONG', 'description': 'Indicates the identifier for the account receiving the funds, crucial for identifying the ultimate beneficiary of the transactio

___

### Generate SQL from retrieved context and user input

Prompt partially adapted from https://python.langchain.com/docs/tutorials/sql_qa/ (langchain-ai/sql-query-system-prompt)

In [23]:
def generate_initial_sql(retrieved_contents: str, query_text: str):
    """
    STEP 1: Generates an initial SQL query from the user query and retrieved knowledge.
    """
    messages = [
        {
            "role": "system",
            "content": (
                "You are an expert in generating SQL queries from natural language queries, especially in the banking industry."
            )
        },
        {
            "role": "user",
            "content": f"""
            Here is the top ranked retrieved relevant data records (based on the
            user's query) from a knowledge graph containing semantic information of a SQL database:
            
            {retrieved_contents}
            
            ---
            Using the above retrieved information of the database, generate a syntactically correct SQL query to help
            find the answer for the user's query: {query_text}.
            
            Do not use any other information or knowledge other than the retrieved information above. Within this retrieved
            information, make sure you rank and use the most appropriate and relevant tables and columns when generating the
            SQL code to answer the user query.
            
            For more context, 'Concept' refers to the linkage of various columns to a single concept or entity.
            It is also useful for you to understand the primary key and foreign key linkages.
            
            Output the final SQL query as plain text (and nothing else).
            """
        }
    ]
    response = client.chat.completions.create(
        messages=messages,
        model=config["LLM_MODEL"],
        temperature=config["TEMPERATURE"],
        seed=config["SEED"]
    )
    sql_output = response.choices[0].message.content
    return sql_output.strip()

In [24]:
# OPTIONAL: Set up double checker of SQL query to validate
def double_check_and_refine_sql(initial_sql: str, query_text: str):
    """
    STEP 2: Double-check the initial SQL to see if it addresses the user's question.
    If needed, refine it. Return the refined SQL (or the same one if it's already good).
    """
    messages = [
        {
            "role": "system",
            "content": (
                "You are a SQL expert. You will receive an initial SQL query and decide if it truly addresses the user's question. "
                "If needed, refine or fix it. Otherwise, confirm it's correct."
            )
        },
        {
            "role": "user",
            "content": f"""
            The user's query is: "{query_text}"
            The initially generated SQL is:
            
            {initial_sql}
            
            Check if:
            1. The SQL is syntactically correct.
            2. The SQL will answer the user's question accurately (based on the user query).
            3. The SQL only uses columns/tables suggested by the database info (no out-of-scope info).
            
            If something is missing or incorrect, refine/fix the SQL. Otherwise, confirm it's already correct.
            
            Output the final, refined SQL as plain text (and nothing else).
            """
        }
    ]
    response = client.chat.completions.create(
        messages=messages,
        model=config["LLM_MODEL"],
        temperature=config["TEMPERATURE"],
        seed=config["SEED"]
    )
    refined_sql = response.choices[0].message.content
    return refined_sql.strip()

In [25]:
def run_sql_generation_pipeline(query_text: str, retrieved_contents: str):
    """
    Orchestrates the 2-step pipeline.
    1) Generate initial SQL
    2) Double-check/refine
    """
    initial_sql = generate_initial_sql(retrieved_contents, query_text)
    refined_sql = double_check_and_refine_sql(initial_sql, query_text)
    print("=== FINAL SQL ===")
    print(refined_sql, "\n")

    return refined_sql

In [26]:
final_sql_query = run_sql_generation_pipeline(query_text, retrieved_contents)

=== FINAL SQL ===
```sql
SELECT d.A2 AS district_name, COUNT(l.account_id) AS customer_count
FROM district d
JOIN loan l ON d.A1 = l.account_id
GROUP BY d.A2
ORDER BY customer_count DESC
LIMIT 1;
``` 

